# Probability Essentials — FAANG-Level Lab

**Goal:** ML-relevant probability: expectation, variance, Bayes, and simulation checks.

**Outcome:** You can reason about uncertainty, distributions, and Bayes updates (interview-ready).


In [1]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Section 1 — Discrete Random Variables

### Task 1.1: Expectation & variance from a PMF
Given values x and probabilities p (sum to 1):
- implement E[X] and Var(X)

# HINT:
- E[X] = sum p_i x_i
- Var(X) = E[X^2] - (E[X])^2

**Explain:** Why is variance not linear, but expectation is?

In [4]:
# Expectation is linear because it's just a weighted sum, no nonlinear terms involved whereas, Variance is not linear because it involves squared terms and covariance.

In [3]:
def expectation(x, p):
    x = np.asarray(x, dtype = float)                    # convert values to float NumPy array
    p = np.asanyarray(p, dtype = float)                 # convert probabilities to float NumPy array
    return float(np.sum(p * x))                         # compute weighted average

def variance(x, p):
    x = np.asarray(x, dtype = float)                    # to float NumPy array
    p = np.asarray(p, dtype = float)                    # to float NumPy array
    ex = np.sum(p * x)                                  # compute expectation E[X]
    ex2 = np.sum(p * x * x)                             # compute second moment E[X²]
    return float(ex2 - ex * ex)                         # compute variance Var(X) = E[X²] - (E[X])²

x = np.array([0, 1, 2])
p = np.array([0.2, 0.5, 0.3])
mu = expectation(x, p)                                  # calculate expected value
var = variance(x, p)                                    # calculate variance
print('E[X]=', mu, 'Var=', var)
check('mu', abs(mu - 1.1) < 1e-9)
check('var', abs(var - (0.2*0 + 0.5*1 + 0.3*4 - 1.1**2)) < 1e-9)    # verify variance formula

E[X]= 1.1 Var= 0.48999999999999977
OK: mu
OK: var


## Section 2 — Conditional Probability & Bayes

### Task 2.1: Bayes theorem (classic interview)
Disease test example:
- prevalence P(D)=0.01
- sensitivity P(+|D)=0.99
- false positive rate P(+|~D)=0.05
Compute P(D|+)

# HINT:
P(D|+) = P(+|D)P(D) / (P(+|D)P(D) + P(+|~D)P(~D))

**FAANG gotcha:** base-rate fallacy.

In [14]:
P_D = 0.01                      # probability of disease
P_pos_given_D = 0.99            # probability test is positive if disease is present
P_pos_given_notD = 0.05         # probability test is positive if disease absent (false positive rate)

# TODO
P_notD = 1 - P_D                # probability of no disease
P_pos = P_pos_given_D * P_D + P_pos_given_notD * P_notD     # total probability of positive test
P_D_given_pos = (P_pos_given_D * P_D) / P_pos               # Bayes theorem: probability disease given positive test
print('P(D|+)=', P_D_given_pos)
check('range', 0 <= P_D_given_pos <= 1)
# Should be around 0.166...
check('approx', abs(P_D_given_pos - (0.99*0.01)/(0.99*0.01 + 0.05*0.99)) < 1e-12)   # verify against analytic value

P(D|+)= 0.16666666666666669
OK: range
OK: approx


### Task 2.2: Simulation check (sanity)
Simulate N people and estimate P(D|+) empirically.

# HINT:
- sample disease ~ Bernoulli(P_D)
- sample test result conditional on disease

**Explain:** Why does simulation converge to the analytic value?

In [ ]:
# Simulation converges to the analytic value because as the number of samples increases, the Large Numbers ensures that sample averages approach true expectations as the number of samples increases.

In [7]:
N = 200000                                                                  # number of simulated individuals
disease = rng.random(N) < P_D                                               # randomly assign disease presence based on base rate P_D
test_pos = np.empty(N, dtype = bool)                                        # create empty boolean array for test results
test_pos[disease] = rng.random(disease.sum()) < P_pos_given_D               # simulate positive test for diseased individuals
test_pos[~disease] = rng.random((~disease).sum()) < P_pos_given_notD        # simulate false positives for healthy individuals

# estimate P(D|+)
est = disease[test_pos].mean()                                              # estimate probability of disease given positive test by averaging
print('estimate', est)
check('close', abs(est - P_D_given_pos) < 0.01)                             # sanity check: simulation close to analytic Bayes probability

estimate 0.16594032852832719
OK: close


## Section 3 — Continuous Distributions (Normal)

### Task 3.1: Standardization (z-score)
Given X ~ Normal(mu, sigma^2). Compute standardized Z=(X-mu)/sigma.

# HINT:
- simulate X and check Z mean ~0, std ~1

**ML link:** standardization shows up in preprocessing and SGD stability.

In [8]:
mu, sigma = 5.0, 2.0                            # set mean and standard deviation for normal distribution
X = rng.normal(mu, sigma, size=200000)          # generate 200,000 samples from N(mu, sigma^2)

Z = (X - mu) / sigma                            # standardize X to have mean 0 and std 1 (Z-scores)
print('Z mean', Z.mean(), 'Z std', Z.std())
check('mean0', abs(Z.mean()) < 0.02)            # verify Z mean is approximately 0
check('std1', abs(Z.std() - 1.0) < 0.02)        # verify Z std is approximately 1

Z mean 0.0007244297435320092 Z std 1.001201581861585
OK: mean0
OK: std1


## Section 4 — Naive Bayes Thinking (Optional Mini)

### Task 4.1: Compute log-odds for a toy Naive Bayes
Given word likelihoods for spam vs ham, compute posterior odds for a message.

# HINT:
- work in log space (sum logs)

**FAANG gotcha:** multiplying small probabilities underflows; use logs.

In [11]:
# Toy params
prior_spam = 0.2                    # prior probability of spam
prior_ham = 0.8                     # prior probability of non-spam
P_word_given_spam = {'free': 0.08, 'win': 0.05, 'meeting': 0.001}       # likelihood of words given spam
P_word_given_ham  = {'free': 0.002, 'win': 0.001, 'meeting': 0.03}      # likelihood of words given ham
message = ['free', 'win']           # words in the message

# compute log posterior ratio log P(spam|msg) - log P(ham|msg) up to constant
log_ratio = np.log(prior_spam) - np.log(prior_ham)                              # log posterior ratio with log priors
for w in message:
    log_ratio += np.log(P_word_given_spam[w]) - np.log(P_word_given_ham[w])     # add log-likelihood ratios for each word
print('log_ratio', log_ratio)
check('finite', np.isfinite(log_ratio))                                         # sanity check: log_ratio is a finite number

log_ratio 6.214608098422191
OK: finite


---
## Submission Checklist
- All TODOs completed
- Checks pass
- Explain prompts answered
